# Task 5 — Fuzzy Text Matching & Standardization

**Plan reference:** `plan.md` Section 2 (Task 5) + Section 4.4
**Primary tool:** RapidFuzz (`fuzz.ratio`)
**Dimension:** Consistency


## Objective

Implement and validate fuzzy text standardization that clusters near-duplicate string values and maps each original spelling to a canonical form — without mutating source data until the caller explicitly applies the mapping.


## Background

Client ERP exports often store the same real-world concept under slightly different spellings:

- Spacing: `New York` vs `NewYork`
- Typos / locale: `colour` vs `color`
- Case variants (handled case-insensitively by default)

Exact case/whitespace normalization (existing `checks/consistency.py`) cannot collapse `colour`/`color`. RapidFuzz similarity can.

> True abbreviations like `Lahore` vs `LHR` score too low on `fuzz.ratio` to merge at the plan default threshold (90). That remains a known limitation — see Discussion.


## Dataset Overview

Synthetic categorical column with planted near-duplicates. No real personal data.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from rapidfuzz import fuzz

repo_root = Path.cwd().resolve()
if not (repo_root / "data_quality_engine").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_quality_engine.config.settings import SETTINGS
from data_quality_engine.engine.standardization.fuzzy_match import (
    apply_standardization,
    check_fuzzy_standardization_frame,
    standardize_frame,
    standardize_values,
)

print("fuzzy_threshold =", SETTINGS["fuzzy_threshold"])
print("fuzzy_case_insensitive =", SETTINGS["fuzzy_case_insensitive"])
print("fuzzy_eligible_roles =", SETTINGS["fuzzy_eligible_roles"])


## Configuration

Thresholds come from `SETTINGS` (plan default 90). Override per-call when experimenting.


In [ ]:
THRESHOLD = SETTINGS["fuzzy_threshold"]
print(f"Using threshold={THRESHOLD}")


## Data Loading


In [ ]:
df = pd.DataFrame(
    {
        "status": ["Paid", "PAID", "paid", "Open", "open", "Closed"],
        "city": ["New York", "NewYork", "Boston", "boston", "Chicago", "Chicago"],
        "amount": [100.0, 110.0, 95.0, 200.0, 105.0, 99.0],
        "notes": [
            "colour preference noted",
            "color preference noted",
            "ship ASAP",
            "ship asap",
            "n/a",
            "follow up",
        ],
    }
)
df


## Data Cleaning

No destructive cleaning here — Task 5 builds a **mapping**. Application is opt-in via `apply_standardization`.


In [ ]:
print("Null counts:")
print(df.isna().sum())
print("Unique status values:", df["status"].nunique())


## Exploratory Data Analysis


In [ ]:
print("status value counts:")
print(df["status"].value_counts())
print()
print("city value counts:")
print(df["city"].value_counts())

vals = sorted(df["status"].dropna().astype(str).unique())
print()
print("Pairwise fuzz.ratio (casefolded):")
for i, a in enumerate(vals):
    for b in vals[i + 1 :]:
        print(f"  {a!r:10} vs {b!r:10} -> {fuzz.ratio(a.casefold(), b.casefold()):5.1f}")


## Methodology

1. Collect unique non-null string values and their frequencies.
2. Seed clusters with the most frequent values first (stable canonicals).
3. Assign each remaining value to a cluster when `fuzz.ratio >= threshold` (case-insensitive by default).
4. Canonical = highest-frequency member of the cluster.
5. Return `{original: canonical}`; caller applies with `.map()` / `apply_standardization`.


## Feature Engineering (if applicable)

Not required — standardization operates on raw string values. Column roles from `classify_columns` gate which columns are eligible.


## Algorithm Selection

| Method | Why chosen / rejected |
|--------|----------------------|
| **RapidFuzz `fuzz.ratio` (chosen)** | plan.md Section 2 Task 5 primary tool; C++ backed; matches TheFuzz accuracy |
| TheFuzz | Runner-up; slower pure-Python |
| Exact casefold only | Already covered by `checks/consistency.py`; cannot merge typos |
| Embedding / LLM | Forbidden in Phase 1 (no AI) |


## Implementation


In [ ]:
roles = {
    "status": "categorical",
    "city": "categorical",
    "amount": "measurement",
    "notes": "free_text",
}

status_map = standardize_values(df["status"], threshold=THRESHOLD)
city_map = standardize_values(df["city"], threshold=THRESHOLD)
print("status mapping:", status_map)
print("city mapping:", city_map)

frame_maps = standardize_frame(df, roles=roles, threshold=THRESHOLD)
print("frame mappings (non-identity only):")
for col, mapping in frame_maps.items():
    changed = {a: b for a, b in mapping.items() if a != b}
    print(f"  {col}: {changed}")


## Intermediate Results


In [ ]:
results = check_fuzzy_standardization_frame(df, roles=roles, threshold=THRESHOLD)
for r in results:
    print(
        f"{r.column:12} status={r.status:7} issues={r.issues_found:3} "
        f"reason={r.details.get('reason', '-')}"
    )
    for c in r.details.get("clusters") or []:
        print(f"    canonical={c['canonical']!r} variants={c['variants']}")

cleaned_status = apply_standardization(df["status"], mapping=status_map)
print()
print("before:", df["status"].tolist())
print("after: ", cleaned_status.tolist())


## Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

before_counts = df["status"].astype(str).value_counts()
after_counts = cleaned_status.astype(str).value_counts()

axes[0].bar(before_counts.index.astype(str), before_counts.values, color="#4C78A8")
axes[0].set_title("Status — before standardization")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(after_counts.index.astype(str), after_counts.values, color="#F58518")
axes[1].set_title("Status — after standardization")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()


## Evaluation


In [ ]:
sweep = []
for thr in [100, 95, 90, 85, 80, 70]:
    m = standardize_values(df["city"], threshold=thr)
    remapped = sum(1 for a, b in m.items() if a != b)
    sweep.append({"threshold": thr, "values_remapped": remapped})

pd.DataFrame(sweep)


## Discussion

- At threshold 90, spacing variants (`New York` / `NewYork`) and case variants collapse reliably.
- Abbreviations (`Lahore` / `LHR`) need a synonym list or a much lower threshold (high false-positive risk).
- Measurement columns are skipped by role — fuzzy merges on amounts would be incorrect.


## Business Insights

Standardizing categorical labels before grouping / joins reduces split categories in dashboards (e.g. Paid vs PAID counted as two payment states).


## Limitations

- `fuzz.ratio` is character-based; semantic synonyms are out of scope for Phase 1.
- Unique-value cap (`fuzzy_max_unique`) drops rare tails on very wide cardinality columns.
- CLI reports mappings but does not rewrite the working DataFrame (explicit apply keeps audits reversible).


## Future Improvements

- Optional synonym dictionaries for known ERP abbreviations.
- Token-sort / partial ratio scorers behind a settings switch.
- Persist mapping tables into the Phase 1 PDF/XLSX report.


## Conclusion

Task 5 delivers plan-compliant RapidFuzz standardization: `standardize_values` returns `{original: canonical}`, checks surface consistency issues, and the pipeline runs it after PII without breaking Tasks 1–4.
